In [ ]:
import pandas as pd
import numpy as np

# -------------------------------------------------------------------
# 1. Load data
# -------------------------------------------------------------------
# Load the first sheet regardless of its name
df = pd.read_excel('../data/task_2_data_ex.xlsx', sheet_name=0)

# Clean plant_id
df['plant_id'] = df['plant_id'].replace(r'^=.*', np.nan, regex=True).ffill()

# Convert types
df['year'] = df['year'].astype(int)
df['month'] = df['month'].astype(int)
for col in ['produced_material_quantity', 'component_material_quantity']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.dropna(subset=['produced_material', 'component_material'], inplace=True)

# -------------------------------------------------------------------
# 2. Helper functions
# -------------------------------------------------------------------
def build_graph(group_df):
    prod_info = {}
    components = {}
    for _, row in group_df.iterrows():
        parent = row['produced_material']
        child = row['component_material']
        if parent not in prod_info:
            prod_info[parent] = {
                'release_type': row['produced_material_release_type'],
                'production_type': row['produced_material_production_type'],
                'quantity': row['produced_material_quantity']
            }
        comp_edge = {
            'component_id': child,
            'release_type': row['component_material_release_type'],
            'production_type': row['component_material_production_type'],
            'consumption_quantity': row['component_material_quantity']
        }
        components.setdefault(parent, []).append(comp_edge)
    return prod_info, components

def process_material(parent, fin_attrs, prod_info, components, results):
    if parent not in components:
        return
    for comp in components[parent]:
        row = {
            'Plant': fin_attrs['plant'],
            'Fin_material_id': fin_attrs['fin_material'],
            'Fin_material_release_type': fin_attrs['fin_release_type'],
            'Fin_material_production_type': fin_attrs['fin_production_type'],
            'Fin_production_quantity': fin_attrs['fin_quantity'],
            'Prod_Material_id': parent,
            'Prod_material_Release_type': prod_info[parent]['release_type'],
            'Prod_material_production_type': prod_info[parent]['production_type'],
            'Prod_material_production_quantity': prod_info[parent]['quantity'],
            'Component_id': comp['component_id'],
            'Component_material_release_type': comp['release_type'],
            'Component_material_production_type': comp['production_type'],
            'Component_consumption_quantity': comp['consumption_quantity'],
            'Year': fin_attrs['year']
        }
        results.append(row)
        # Recurse if component is a PROD material with its own production info
        if (comp['component_id'] in prod_info and 
            prod_info[comp['component_id']]['release_type'] == 'PROD'):
            process_material(comp['component_id'], fin_attrs, prod_info, components, results)

def explode_group(group_df, plant, year, month):
    prod_info, components = build_graph(group_df)
    results = []
    fin_rows = group_df[group_df['produced_material_release_type'] == 'FIN']
    for _, fin_row in fin_rows.iterrows():
        fin_id = fin_row['produced_material']
        fin_attrs = {
            'plant': plant,
            'year': year,
            'fin_material': fin_id,
            'fin_release_type': 'FIN',
            'fin_production_type': fin_row['produced_material_production_type'],
            'fin_quantity': fin_row['produced_material_quantity']
        }
        if fin_id in components:
            for child_edge in components[fin_id]:
                child = child_edge['component_id']
                # Only start recursion if the child is a PROD material (has its own production)
                if child in prod_info and prod_info[child]['release_type'] == 'PROD':
                    process_material(child, fin_attrs, prod_info, components, results)
    return results

# -------------------------------------------------------------------
# 3. Process all groups
# -------------------------------------------------------------------
all_results = []
for (plant, year, month), group in df.groupby(['plant_id', 'year', 'month']):
    res = explode_group(group, plant, year, month)
    all_results.extend(res)

output_df = pd.DataFrame(all_results)

# -------------------------------------------------------------------
# 4. Finalize columns and case
# -------------------------------------------------------------------
final_columns = [
    'Plant', 'Fin_material_id', 'Fin_material_release_type', 'Fin_material_production_type',
    'Fin_production_quantity', 'Prod_Material_id', 'Prod_material_Release_type',
    'Prod_material_production_type', 'Prod_material_production_quantity',
    'Component_id', 'Component_material_release_type', 'Component_material_production_type',
    'Component_consumption_quantity', 'Year'
]
output_df = output_df[final_columns]
output_df.columns = [c.lower() for c in output_df.columns]

# Save
output_path = '../output/bom_explosion_result.xlsx'
output_df.to_excel(output_path, index=False, engine='openpyxl', sheet_name='BoM_Explosion')

print(f"Done. Results saved to {output_path}")
print("Output shape:", output_df.shape)